In [1]:
import pandas as pd

In [2]:
df_X = pd.read_csv("data/X_data.csv")
df_Y = pd.read_csv("data/Y_data.csv")

In [3]:
df_X.head(5)

,recordid,SAPS-I,SOFA,Age,Gender,Height,Weight,CCU,CSRU,SICU,...,Platelets_last,TroponinI_last,TroponinT_last,WBC_last,Weight_last,pH_last,MechVentStartTime,MechVentDuration,MechVentLast8Hour,UrineOutputSum
0,137517,-1,2,56.0,0.0,NaN,79.6,0,0,0,...,444.0,NaN,NaN,7.3,79.6,NaN,NaN,NaN,NaN,NaN
1,145680,10,3,72.0,1.0,NaN,70.0,0,0,0,...,82.0,NaN,0.07,2.4,70.0,7.48,NaN,NaN,NaN,NaN
2,138649,-1,8,81.0,0.0,NaN,NaN,0,0,0,...,211.0,NaN,NaN,7.0,NaN,NaN,NaN,NaN,NaN,NaN
3,149075,16,8,56.0,1.0,180.3,94.8,1,0,0,...,204.0,NaN,NaN,11.4,98.9,7.44,1128.0,470.0,0.0,4.0
4,141408,14,7,52.0,1.0,182.9,120.6,0,1,0,...,352.0,NaN,NaN,9.0,120.6,7.43,143.0,2580.0,1.0,16.0


In [4]:
df_Y.head(5)

,In-hospital_death
0,0
1,0
2,0
3,0
4,0


In [5]:
df_X.shape

(3600, 120)

In [6]:
df_Y.shape

(3600, 1)

In [7]:
from sklearn.model_selection import train_test_split

In [8]:
df_X = df_X.drop('recordid', axis = 1)
X_train, X_test, Y_train, Y_test = train_test_split(df_X, df_Y, test_size=0.3, random_state=42, stratify=df_Y)

In [9]:
missing = round(X_train.isnull().sum()/len(df_X), 2).sort_values(ascending=False)
missing[missing > 0]

TroponinI_last       0.66
TroponinI_first      0.66
Cholesterol_first    0.64
Cholesterol_last     0.64
TroponinT_first      0.55
                     ... 
HCT_last             0.01
HCO3_last            0.01
Platelets_last       0.01
Temp_median          0.01
Na_last              0.01
Length: 112, dtype: float64

Nijedna kolona nije u potpunosti popunjena nedostajućim vrednostima. Pošto se ovde radi o medicinskim nalazima nećemo odbacivati kolone sa dosta nedostajućih vrednosti jer je verovatno taj nalaz urađen za tu osobu baš zbog sumnje doktora ili neke poznate ranije dijagnoze (retki nalazi nisu nasumično radjeni nego baš nasuprot ciljno) što svakako može da bude vredna informacija. Zato ćemo nedostajuće vrednosti zameniti medijanom i dodaćemo kolonu koja će nositi informaciju o tome da li je vrednost nedostajuća (ovo samo za redje nalaze) ili ne. Indikatorske kolone dodaćemo samo za kolone koje imaju preko 60% nedostajućih vrednosti jer njih smatramo retkima (nalaz se ne pojavljuje kod većine pacijenata)

In [10]:
def fillna_and_add_indicator_columns(X):
    rare_threshold = 0.6
    indicator_columns = {}
    for col in missing.index:
        if missing[col] > rare_threshold:
            indicator_columns[col + '_missing'] = X[col].isnull().astype(int)
    X = pd.concat([X, pd.DataFrame(indicator_columns, index=X.index)], axis=1)
    X = X.fillna(X_train.median())
    return X

In [11]:
X_train2 = X_train.dropna(thresh=0.1*len(X_train), axis=1)
X_test2 = X_test.dropna(thresh=0.1*len(X_test), axis=1)
X_train2.fillna(X_train2.median(), inplace=True)
X_test2.fillna(X_test2.median(), inplace=True)
print(X_train2.shape, X_test2.shape)

X_train = fillna_and_add_indicator_columns(X_train)
X_test = fillna_and_add_indicator_columns(X_test)

(2520, 115) (1080, 115)


In [12]:
from sklearn.decomposition import PCA
import numpy as np

In [13]:
X_train_std = (X_train - X_train.mean()) / X_train.std()
X_test_std = (X_test - X_train.mean()) / X_train.std()

X_test2 = (X_test2 - X_train2.mean()) / X_train2.std()
X_train2 = (X_train2 - X_train2.mean()) / X_train2.std()

In [14]:
X_train_std.to_csv("data/processed/X_train0.csv", index=False)
X_test_std.to_csv("data/processed/X_test0.csv", index=False)

X_train2.to_csv("data/processed/X_train02.csv", index=False)
X_test2.to_csv("data/processed/X_test02.csv", index=False)

In [15]:
pca90 = PCA(n_components=0.90)
pca95 = PCA(n_components=0.95)

pca90.fit(X_train_std)
pca95.fit(X_train_std)

print(f"Components: {len(pca90.components_)}")
print(f"Components: {len(pca95.components_)}")

Components: 59
Components: 74


In [16]:
X_train90 = pca90.transform(X_train_std)
X_test90 = pca90.transform(X_test_std)

X_train95 = pca95.transform(X_train_std)
X_test95 = pca95.transform(X_test_std)

Logističkom regresijom ćemo proveriti razliku u kvalitetu klasifikacije pri izmedju ova 2 skupa

In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

In [18]:
lr = LogisticRegression()
lr.fit(X_train90, Y_train.values.ravel())

print(f"Accuracy for pca90: {accuracy_score(Y_test.values.ravel(), lr.predict(X_test90))}")
print(f"F1 Score for pca90: {f1_score(Y_test.values.ravel(), lr.predict(X_test90))}")

lr.fit(X_train95, Y_train.values.ravel())
print(f"Accuracy for pca95: {accuracy_score(Y_test.values.ravel(), lr.predict(X_test95))}")
print(f"F1 Score for pca95: {f1_score(Y_test.values.ravel(), lr.predict(X_test95))}")


Accuracy for pca90: 0.8712962962962963
F1 Score for pca90: 0.4034334763948498
Accuracy for pca95: 0.8722222222222222
F1 Score for pca95: 0.4152542372881356


In [19]:
Y_train.shape

(2520, 1)

F1 je blago bolji kod pca95 pa pošto je u pitanju predvidjanje smrtnosti, koristićemo pca95 jer je malo bolji sa ovako nebalansiranom klasom (dakle model je na tom skupu bolje naučio da predvidja obe klase). Pored pca95 koristicemo i pca sa 50 komponenti čisto radi poredjenja.

In [20]:
pca50 = PCA(n_components=50)
pca50.fit(X_train_std)

X_train50 = pca50.transform(X_train_std)
X_test50 = pca50.transform(X_test_std)

print(f"Explained variance ratio: {len(pca50.explained_variance_ratio_)}")

pd.DataFrame(X_train50, index=X_train.index, columns=[f"PCA_{i}" for i in range(X_train50.shape[1])]).to_csv("data/processed/X_train50.csv", index=False)
pd.DataFrame(X_test50, index=X_test.index, columns=[f"PCA_{i}" for i in range(X_test50.shape[1])]).to_csv("data/processed/X_test50.csv", index=False)

Explained variance ratio: 50


In [21]:
pd.DataFrame(X_train95, index=X_train.index, columns=[f"PCA_{i}" for i in range(X_train95.shape[1])]).to_csv("data/processed/X_train95.csv", index=False)
pd.DataFrame(X_test95, index=X_test.index, columns=[f"PCA_{i}" for i in range(X_test95.shape[1])]).to_csv("data/processed/X_test95.csv", index=False)
Y_train.to_csv("data/processed/Y_train.csv", index=False)
Y_test.to_csv("data/processed/Y_test.csv", index=False)